# Sync Author Display-Name Curations

Syncs display_name curations from the users Heroku Postgres database to a local Databricks Delta table. Runs in `jobs/authors.yaml` upstream of `ApplyAuthorNameCurations` and `CreateAuthors`.

**Source** (typed view over `openalex_users.public.curations`):
- `author_display_name_curations` — set a display_name for an author profile

**Target**:
- `openalex.authors.author_names_curations` — one row per curated author, `(author_id, curated_display_name, updated_datetime)`

When multiple curators have set display_name for the same author, latest-wins by source `created` timestamp.

The curated display_name is applied as an **override** in `CreateAuthors`, not written in-place to `openalex.authors.authors`. Deleting a curation from the Heroku source causes the MERGE's `WHEN NOT MATCHED BY SOURCE THEN DELETE` to remove it here, and the next `CreateAuthors` run falls back to the organic name.

## Create target table (idempotent)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS openalex.authors.author_names_curations (
    author_id             BIGINT NOT NULL,
    curated_display_name  STRING,
    updated_datetime      TIMESTAMP
)
USING DELTA
CLUSTER BY (author_id)

In [ ]:
%sql
ALTER TABLE openalex.authors.author_names_curations DROP COLUMN IF EXISTS curated_full_name

## Sync names curations

In [ ]:
%sql
SELECT author_id, new_display_name AS curated_display_name
FROM (
  SELECT
    author_id,
    new_display_name,
    ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
  FROM openalex_users.public.author_display_name_curations
)
WHERE rn = 1

In [ ]:
%sql
MERGE INTO openalex.authors.author_names_curations AS target
USING (
  SELECT author_id, curated_display_name, CURRENT_TIMESTAMP() AS updated_datetime
  FROM (
    SELECT
      author_id,
      new_display_name AS curated_display_name,
      ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
    FROM openalex_users.public.author_display_name_curations
  )
  WHERE rn = 1
) AS source
ON target.author_id = source.author_id
WHEN MATCHED THEN UPDATE SET
  target.curated_display_name = source.curated_display_name,
  target.updated_datetime     = source.updated_datetime
WHEN NOT MATCHED THEN INSERT (author_id, curated_display_name, updated_datetime)
  VALUES (source.author_id, source.curated_display_name, source.updated_datetime)
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Verify sync results

In [ ]:
%sql
SELECT
  COUNT(*)                    AS total_curated_authors,
  COUNT(curated_display_name) AS total_display_names,
  MAX(updated_datetime)       AS last_sync
FROM openalex.authors.author_names_curations

In [ ]:
%sql
SELECT * FROM openalex.authors.author_names_curations
ORDER BY updated_datetime DESC
LIMIT 100